In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/muhammadumer7804/covid-19-dataset/covid_19_clean_complete.csv
/kaggle/input/datasets/muhammadumer7804/covid-19-dataset/country_wise_latest.csv
/kaggle/input/datasets/muhammadumer7804/covid-19-dataset/worldometer_data.csv


# **Step 1: Import Libraries**

In [2]:
# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# set default plot style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('All libraries imported successfully!')

All libraries imported successfully!


# **Step 2: Load Data**

In [3]:
# Load All Three Datasets

path = '/kaggle/input/datasets/muhammadumer7804/covid-19-dataset/'

# time series data
df_complete = pd.read_csv(path + 'covid_19_clean_complete.csv')

# country wise latest data
df_country = pd.read_csv(path + 'country_wise_latest.csv')

# world summary data
df_world = pd.read_csv(path + 'worldometer_data.csv')

print('All datasets loaded!')
print()
print('covid_19_clean_complete :', df_complete.shape)
print('country_wise_latest     :', df_country.shape)
print('worldometer_data        :', df_world.shape)

All datasets loaded!

covid_19_clean_complete : (49068, 10)
country_wise_latest     : (187, 15)
worldometer_data        : (209, 16)


# **Step 3: First Look**

In [4]:
#  First Look at All Datasets

print('=== COVID Complete Dataset ===')
print(df_complete.head())
print()
print('Columns:', df_complete.columns.tolist())
print()
print('=== Country Wise Latest ===')
print(df_country.head())
print()
print('=== Worldometer Data ===')
print(df_world.head())

=== COVID Complete Dataset ===
  Province/State Country/Region       Lat       Long        Date  Confirmed  \
0            NaN    Afghanistan  33.93911  67.709953  2020-01-22          0   
1            NaN        Albania  41.15330  20.168300  2020-01-22          0   
2            NaN        Algeria  28.03390   1.659600  2020-01-22          0   
3            NaN        Andorra  42.50630   1.521800  2020-01-22          0   
4            NaN         Angola -11.20270  17.873900  2020-01-22          0   

   Deaths  Recovered  Active             WHO Region  
0       0          0       0  Eastern Mediterranean  
1       0          0       0                 Europe  
2       0          0       0                 Africa  
3       0          0       0                 Europe  
4       0          0       0                 Africa  

Columns: ['Province/State', 'Country/Region', 'Lat', 'Long', 'Date', 'Confirmed', 'Deaths', 'Recovered', 'Active', 'WHO Region']

=== Country Wise Latest ===
  Country/R

# **Step 4: Check Missing Values**

In [5]:
# Check Missing Values

print('=== covid_19_clean_complete ===')
print(df_complete.isnull().sum())

print()
print('=== country_wise_latest ===')
print(df_country.isnull().sum())

print()
print('=== worldometer_data ===')
print(df_world.isnull().sum())

=== covid_19_clean_complete ===
Province/State    34404
Country/Region        0
Lat                   0
Long                  0
Date                  0
Confirmed             0
Deaths                0
Recovered             0
Active                0
WHO Region            0
dtype: int64

=== country_wise_latest ===
Country/Region            0
Confirmed                 0
Deaths                    0
Recovered                 0
Active                    0
New cases                 0
New deaths                0
New recovered             0
Deaths / 100 Cases        0
Recovered / 100 Cases     0
Deaths / 100 Recovered    0
Confirmed last week       0
1 week change             0
1 week % increase         0
WHO Region                0
dtype: int64

=== worldometer_data ===
Country/Region        0
Continent             1
Population            1
TotalCases            0
NewCases            205
TotalDeaths          21
NewDeaths           206
TotalRecovered        4
NewRecovered        206
ActiveCases

# **Step 5: Handle Missing Values**

In [6]:
#Handle Missing Values

# covid_19_clean_complete
# Province/State — not needed, drop this column
df_complete.drop(columns=['Province/State'], inplace=True)

# worldometer_data
# NewCases, NewDeaths, NewRecovered — almost all missing, drop these columns
df_world.drop(columns=['NewCases', 'NewDeaths', 'NewRecovered'], inplace=True)

# fill remaining missing values with 0
df_world.fillna(0, inplace=True)

print('Missing values handled!')
print()
print('df_complete missing:', df_complete.isnull().sum().sum())
print('df_country missing :', df_country.isnull().sum().sum())
print('df_world missing   :', df_world.isnull().sum().sum())

Missing values handled!

df_complete missing: 0
df_country missing : 0
df_world missing   : 0


# **Step 6: Fix Data Types**

In [7]:
# Fix Data Types
# Date column is object, convert to datetime

df_complete['Date'] = pd.to_datetime(df_complete['Date'])

print('Data types fixed!')
print()
print('Date column type:', df_complete['Date'].dtype)
print()
print('Date range:')
print('From:', df_complete['Date'].min())
print('To  :', df_complete['Date'].max())

Data types fixed!

Date column type: datetime64[ns]

Date range:
From: 2020-01-22 00:00:00
To  : 2020-07-27 00:00:00


# **Step 7: Check Duplicates**

In [8]:
# Check Duplicates

print('df_complete duplicates:', df_complete.duplicated().sum())
print('df_country duplicates :', df_country.duplicated().sum())
print('df_world duplicates   :', df_world.duplicated().sum())

# remove if any
df_complete.drop_duplicates(inplace=True)
df_country.drop_duplicates(inplace=True)
df_world.drop_duplicates(inplace=True)

print()
print('Duplicates removed!')

df_complete duplicates: 0
df_country duplicates : 0
df_world duplicates   : 0

Duplicates removed!
